<a href="https://colab.research.google.com/github/SACHHH05/Text2SQL-LoRA/blob/main/text2sql_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Data
Load the dataset, split it into train/validation/test, and build the prompt format.

In [1]:
!pip install -q datasets
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/text2sql-lora'
DATA_DIR = f'{PROJECT_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)
print('Saving to:', DATA_DIR)

Mounted at /content/drive
Saving to: /content/drive/MyDrive/text2sql-lora/data


In [2]:
from datasets import load_dataset

ds = load_dataset('b-mc2/sql-create-context', split='train')
print(ds)
print(ds[0])

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Dataset({
    features: ['answer', 'question', 'context'],
    num_rows: 78577
})
{'answer': 'SELECT COUNT(*) FROM head WHERE age > 56', 'question': 'How many heads of the departments are older than 56 ?', 'context': 'CREATE TABLE head (age INTEGER)'}


In [3]:
import random
random.seed(0)
for i in random.sample(range(len(ds)), 3):
    ex = ds[i]
    print('QUESTION:', ex['question'])
    print('SCHEMA  :', ex['context'])
    print('SQL     :', ex['answer'])
    print('-' * 60)

QUESTION: What chart positions did the album reach when ep was the format?
SCHEMA  : CREATE TABLE table_name_1 (chart_positions VARCHAR, format VARCHAR)
SQL     : SELECT chart_positions FROM table_name_1 WHERE format = "ep"
------------------------------------------------------------
QUESTION: What is the average attendance for matches where the home team was vida?
SCHEMA  : CREATE TABLE table_name_2 (attendance INTEGER, home VARCHAR)
SQL     : SELECT AVG(attendance) FROM table_name_2 WHERE home = "vida"
------------------------------------------------------------
QUESTION: what is the minimum bush # with kerry # being 3938
SCHEMA  : CREATE TABLE table_13606924_1 (bush__number INTEGER, kerry__number VARCHAR)
SQL     : SELECT MIN(bush__number) FROM table_13606924_1 WHERE kerry__number = 3938
------------------------------------------------------------


In [4]:
ds = ds.shuffle(seed=42)

N_TRAIN, N_VAL, N_TEST = 8000, 200, 300
test_ds  = ds.select(range(0, N_TEST))
val_ds   = ds.select(range(N_TEST, N_TEST + N_VAL))
train_ds = ds.select(range(N_TEST + N_VAL, N_TEST + N_VAL + N_TRAIN))
print(len(train_ds), len(val_ds), len(test_ds))

8000 200 300


In [5]:
def build_prompt(question, context):
    return (
        "You are a SQL expert. Given the table schema and a question, write one SQL query.\n\n"
        f"### Schema:\n{context}\n\n"
        f"### Question:\n{question}\n\n"
        "### SQL:\n"
    )

def to_record(ex):
    prompt = build_prompt(ex['question'], ex['context'])
    return {'prompt': prompt, 'completion': ex['answer'], 'text': prompt + ex['answer']}

train = [to_record(e) for e in train_ds]
val   = [to_record(e) for e in val_ds]
test  = [to_record(e) for e in test_ds]
print(train[0]['text'])

You are a SQL expert. Given the table schema and a question, write one SQL query.

### Schema:
CREATE TABLE table_name_41 (place__posición_ INTEGER, points__pts_ VARCHAR, played__pj_ VARCHAR)

### Question:
What was the top place with over 24 points and more than 14 played?

### SQL:
SELECT MAX(place__posición_) FROM table_name_41 WHERE points__pts_ > 24 AND played__pj_ > 14


In [6]:
import numpy as np
lengths = [len(r['text'].split()) for r in train]
print('words  mean/95th/max:', int(np.mean(lengths)), int(np.percentile(lengths, 95)), max(lengths))

words  mean/95th/max: 52 69 119


In [7]:
import json
def save(name, rows):
    with open(f'{DATA_DIR}/{name}.jsonl', 'w') as f:
        for r in rows:
            f.write(json.dumps(r) + '\n')
    print('saved', name, len(rows))

save('train', train); save('val', val); save('test', test)

saved train 8000
saved val 200
saved test 300


# 2. Baseline
Score the untuned 1.5B model on the test set.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [8]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U transformers accelerate datasets

name, memory.total [MiB]
Tesla T4, 15360 MiB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 18.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [9]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/text2sql-lora'
DATA_DIR = f'{PROJECT_DIR}/data'
RESULTS_DIR = f'{PROJECT_DIR}/results'
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
assert os.path.exists(f'{DATA_DIR}/test.jsonl'), 'Run 01 data first!'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
%%writefile {PROJECT_DIR}/utils.py
import json, re, sqlite3

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

def clean_generation(text):
    """Keep only the first SQL statement the model wrote."""
    text = text.strip()
    for stop in ["###", "\n\n"]:
        if stop in text:
            text = text.split(stop)[0]
    if ";" in text:
        text = text.split(";")[0]
    return text.strip()

def normalize(sql):
    sql = sql.strip().rstrip(";").lower()
    sql = sql.replace('"', "'").replace("`", "")
    sql = re.sub(r"\s+", " ", sql)
    sql = re.sub(r"\s*([(),=<>])\s*", r"\1", sql)
    return sql.strip()

def runs_on_sqlite(sql, schema):
    """Create empty tables from the schema, then try to execute the query."""
    try:
        con = sqlite3.connect(":memory:")
        con.executescript(schema)
        con.execute(sql).fetchall()
        return True
    except Exception:
        return False
    finally:
        try: con.close()
        except Exception: pass

def score(preds, rows):
    em = sum(normalize(p) == normalize(r["completion"]) for p, r in zip(preds, rows))
    ok = sum(runs_on_sqlite(p, r["prompt"].split("### Schema:\n")[1].split("\n\n### Question:")[0])
             for p, r in zip(preds, rows))
    n = len(rows)
    return {"n": n, "exact_match": em / n, "runs_on_sqlite": ok / n}

Writing /content/drive/MyDrive/text2sql-lora/utils.py


In [11]:
import sys
sys.path.insert(0, PROJECT_DIR)
import importlib, utils
importlib.reload(utils)
from utils import *

test = load_jsonl(f'{DATA_DIR}/test.jsonl')
print(len(test), 'test examples')

300 test examples


In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = 'Qwen/Qwen2.5-Coder-1.5B'
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='cuda')
model.eval()
print(f'Parameters: {model.num_parameters()/1e9:.2f}B | GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Parameters: 1.54B | GPU memory used: 3.1 GB


In [13]:
from tqdm.auto import tqdm

@torch.no_grad()
def generate_sql(model, tok, rows, batch_size=16, max_new_tokens=128):
    prompts = [r['prompt'] for r in rows]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outputs = [None] * len(prompts)
    for s in tqdm(range(0, len(order), batch_size)):
        idx = order[s:s + batch_size]
        batch = tok([prompts[i] for i in idx], return_tensors='pt', padding=True).to('cuda')
        gen = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(gen[:, batch['input_ids'].shape[1]:], skip_special_tokens=True)
        for i, t in zip(idx, texts):
            outputs[i] = clean_generation(t)
    return outputs

preds = generate_sql(model, tok, test)

  0%|          | 0/19 [00:00<?, ?it/s]

In [14]:
baseline = score(preds, test)
print(baseline)

{'n': 300, 'exact_match': 0.49666666666666665, 'runs_on_sqlite': 0.95}


In [15]:
shown = 0
for p, r in zip(preds, test):
    if normalize(p) != normalize(r['completion']):
        q = r['prompt'].split('### Question:\n')[1].split('\n\n### SQL:')[0]
        print('Q       :', q)
        print('EXPECTED:', r['completion'])
        print('GOT     :', p)
        print('-' * 60)
        shown += 1
        if shown == 5: break

Q       : Which Points have Touchdowns larger than 0, and an Extra points smaller than 0?
EXPECTED: SELECT SUM(points) FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0
GOT     : SELECT points FROM table_name_70 WHERE touchdowns > 0 AND extra_points < 0
------------------------------------------------------------
Q       : What is the name of the player who is Sco and moving to greenock morton in the summer?
EXPECTED: SELECT name FROM table_name_83 WHERE nat = "sco" AND transfer_window = "summer" AND moving_to = "greenock morton"
GOT     : SELECT name FROM table_name_83 WHERE moving_to = 'greenock morton' AND nat = 'Sco' AND transfer_window = 'summer'
------------------------------------------------------------
Q       : Of all the contestants who got voted, what is the contestant number and name of the contestant who got least votes?
EXPECTED: SELECT T1.contestant_number, T1.contestant_name FROM contestants AS T1 JOIN votes AS T2 ON T1.contestant_number = T2.contestant_numb

In [16]:
import json
with open(f'{RESULTS_DIR}/baseline.json', 'w') as f:
    json.dump({'model': MODEL, **baseline}, f, indent=2)
with open(f'{RESULTS_DIR}/baseline_preds.json', 'w') as f:
    json.dump(preds, f)
print('saved')

saved


In [21]:
!pip uninstall -y torchao
!pip install -q -U transformers accelerate datasets peft
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
PROJECT_DIR = '/content/drive/MyDrive/text2sql-lora'
DATA_DIR = f'{PROJECT_DIR}/data'
RESULTS_DIR = f'{PROJECT_DIR}/results'
ADAPTER_DIR = f'{PROJECT_DIR}/adapters/lora_r16'
os.makedirs(ADAPTER_DIR, exist_ok=True)
assert os.path.exists(f'{PROJECT_DIR}/utils.py'), 'Run notebook 2 first!'

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
sys.path.insert(0, PROJECT_DIR)
from utils import *

train = load_jsonl(f'{DATA_DIR}/train.jsonl')
val   = load_jsonl(f'{DATA_DIR}/val.jsonl')
test  = load_jsonl(f'{DATA_DIR}/test.jsonl')

MODEL = 'Qwen/Qwen2.5-Coder-1.5B'
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='cuda')
print(len(train), len(val), len(test))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

8000 200 300


In [23]:
from peft import LoraConfig, get_peft_model

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora_cfg)

for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [24]:
MAX_LEN = 384

def encode(r):
    p = tok(r['prompt'], add_special_tokens=False)['input_ids']
    c = tok(r['completion'], add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
    return {'input_ids': p + c, 'labels': [-100] * len(p) + c}

train_enc = [e for e in map(encode, train) if len(e['input_ids']) <= MAX_LEN]
val_enc   = [e for e in map(encode, val)   if len(e['input_ids']) <= MAX_LEN]
print('kept', len(train_enc), 'of', len(train), 'train examples')

def collate(batch):
    m = max(len(b['input_ids']) for b in batch)
    pad = tok.pad_token_id
    return {
        'input_ids': torch.tensor([b['input_ids'] + [pad] * (m - len(b['input_ids'])) for b in batch]),
        'attention_mask': torch.tensor([[1] * len(b['input_ids']) + [0] * (m - len(b['input_ids'])) for b in batch]),
        'labels': torch.tensor([b['labels'] + [-100] * (m - len(b['labels'])) for b in batch]),
    }

kept 8000 of 8000 train examples


In [26]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=f'{PROJECT_DIR}/checkpoints',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=15,
    fp16=True,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='no',
    report_to='none',
    remove_unused_columns=False,
    optim='adamw_torch',
)

model.config.use_cache = False
trainer = Trainer(model=model, args=args, train_dataset=train_enc,
                  eval_dataset=val_enc, data_collator=collate)
trainer.train()

Step,Training Loss,Validation Loss
100,0.058823,0.061467
200,0.046913,0.053373
300,0.038193,0.044007
400,0.041018,0.040684
500,0.027151,0.039945


TrainOutput(global_step=500, training_loss=0.05445849347114563, metrics={'train_runtime': 668.209, 'train_samples_per_second': 11.972, 'train_steps_per_second': 0.748, 'total_flos': 7473286478168064.0, 'train_loss': 0.05445849347114563, 'epoch': 1.0})

In [27]:
model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
print('saved adapter to', ADAPTER_DIR)

saved adapter to /content/drive/MyDrive/text2sql-lora/adapters/lora_r16


In [28]:
from tqdm.auto import tqdm
tok.padding_side = 'left'
model.config.use_cache = True
model.eval()

@torch.no_grad()
def generate_sql(model, tok, rows, batch_size=16, max_new_tokens=128):
    prompts = [r['prompt'] for r in rows]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outputs = [None] * len(prompts)
    for s in tqdm(range(0, len(order), batch_size)):
        idx = order[s:s + batch_size]
        batch = tok([prompts[i] for i in idx], return_tensors='pt', padding=True).to('cuda')
        gen = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(gen[:, batch['input_ids'].shape[1]:], skip_special_tokens=True)
        for i, t in zip(idx, texts):
            outputs[i] = clean_generation(t)
    return outputs

preds = generate_sql(model, tok, test)
tuned = score(preds, test)

  0%|          | 0/19 [00:00<?, ?it/s]

In [29]:
base = json.load(open(f'{RESULTS_DIR}/baseline.json'))
print(f"{'':16}{'baseline':>10}{'LoRA r=16':>12}")
print(f"{'exact match':16}{base['exact_match']:>10.1%}{tuned['exact_match']:>12.1%}")
print(f"{'runs on SQLite':16}{base['runs_on_sqlite']:>10.1%}{tuned['runs_on_sqlite']:>12.1%}")

with open(f'{RESULTS_DIR}/lora_r16.json', 'w') as f:
    json.dump({'model': MODEL, 'r': 16, 'alpha': 32, 'lr': 2e-4, 'epochs': 1, **tuned}, f, indent=2)
with open(f'{RESULTS_DIR}/lora_r16_preds.json', 'w') as f:
    json.dump(preds, f)

                  baseline   LoRA r=16
exact match          49.7%       81.3%
runs on SQLite       95.0%       98.0%


# 4. Mistake analysis
Look at which test questions the model gets wrong and why.

In [30]:
from google.colab import drive
drive.mount('/content/drive')
import sys, json
PROJECT_DIR = '/content/drive/MyDrive/text2sql-lora'
sys.path.insert(0, PROJECT_DIR)
from utils import *

test  = load_jsonl(f'{PROJECT_DIR}/data/test.jsonl')
preds = json.load(open(f'{PROJECT_DIR}/results/lora_r16_preds.json'))
base_preds = json.load(open(f'{PROJECT_DIR}/results/baseline_preds.json'))
print(len(test), len(preds), len(base_preds))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
300 300 300


In [31]:
def schema_of(r):
    return r['prompt'].split('### Schema:\n')[1].split('\n\n### Question:')[0]

def question_of(r):
    return r['prompt'].split('### Question:\n')[1].split('\n\n### SQL:')[0]

broken, differs = [], []
for p, r in zip(preds, test):
    if normalize(p) == normalize(r['completion']):
        continue
    (differs if runs_on_sqlite(p, schema_of(r)) else broken).append((p, r))

print(f'{len(broken)} queries do NOT run (real errors)')
print(f'{len(differs)} queries run but differ from the reference')

def show(items, n):
    for p, r in items[:n]:
        print('Q       :', question_of(r))
        print('EXPECTED:', r['completion'])
        print('GOT     :', p)
        print('-' * 60)

print('\n===== BROKEN QUERIES =====')
show(broken, 6)
print('\n===== RUN BUT DIFFER =====')
show(differs, 12)

3 queries do NOT run (real errors)
53 queries run but differ from the reference

===== BROKEN QUERIES =====
Q       : what is the total number of kickoff [a ] where week is week
EXPECTED: SELECT COUNT(kickoff_)[a_] FROM table_11406866_2 WHERE "week" = "week"
GOT     : SELECT COUNT(a_) FROM table_11406866_2 WHERE kickoff_[a_] = "Week"
------------------------------------------------------------
Q       : How many Bangladeshis were admitted when 714 Nepalis and 13,575 Pakistanis were admitted?
EXPECTED: SELECT AVG(bangladeshis_admitted) FROM table_name_23 WHERE nepalis_admitted = 714 AND pakistanis_admitted > 13 OFFSET 575
GOT     : SELECT SUM(bangladeshis_admitted) FROM table_name_23 WHERE nepalis_admitted = 714 AND pakistanis_admitted = 13 OFFSET 575
------------------------------------------------------------
Q       : What is the lowest area when the density is greater than 234.77, the population is less than 965,040, and the rank is 32?
EXPECTED: SELECT MIN(area) FROM table_name_75 

In [32]:
fixed = broke = 0
for b, p, r in zip(base_preds, preds, test):
    b_ok = normalize(b) == normalize(r['completion'])
    p_ok = normalize(p) == normalize(r['completion'])
    fixed += (not b_ok) and p_ok
    broke += b_ok and (not p_ok)
print(f'LoRA fixed {fixed} questions the baseline got wrong')
print(f'LoRA broke {broke} questions the baseline got right')

LoRA fixed 103 questions the baseline got wrong
LoRA broke 8 questions the baseline got right


# 5. Experiments (rank and learning rate)
Train with different LoRA ranks and learning rates and compare results.

# 6. QLoRA 7B
Fine-tune the 7B model in 4-bit (QLoRA) and compare it with the 1.5B LoRA model.

# 3. LoRA r=16
Fine-tune the 1.5B model with LoRA and compare it to the baseline.

In [1]:
!pip uninstall -y torchao
!pip install -q -U transformers accelerate datasets peft
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, gc, glob
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
PROJECT_DIR = '/content/drive/MyDrive/text2sql-lora'
DATA_DIR = f'{PROJECT_DIR}/data'
RESULTS_DIR = f'{PROJECT_DIR}/results'
sys.path.insert(0, PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, set_seed
from peft import LoraConfig, get_peft_model
from utils import *

MODEL = 'Qwen/Qwen2.5-Coder-1.5B'
train = load_jsonl(f'{DATA_DIR}/train.jsonl')
val   = load_jsonl(f'{DATA_DIR}/val.jsonl')
test  = load_jsonl(f'{DATA_DIR}/test.jsonl')

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'left'

MAX_LEN = 384
def encode(r):
    p = tok(r['prompt'], add_special_tokens=False)['input_ids']
    c = tok(r['completion'], add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
    return {'input_ids': p + c, 'labels': [-100] * len(p) + c}

train_enc = [e for e in map(encode, train) if len(e['input_ids']) <= MAX_LEN]
val_enc   = [e for e in map(encode, val)   if len(e['input_ids']) <= MAX_LEN]

def collate(batch):
    m = max(len(b['input_ids']) for b in batch)
    pad = tok.pad_token_id
    return {
        'input_ids': torch.tensor([b['input_ids'] + [pad] * (m - len(b['input_ids'])) for b in batch]),
        'attention_mask': torch.tensor([[1] * len(b['input_ids']) + [0] * (m - len(b['input_ids'])) for b in batch]),
        'labels': torch.tensor([b['labels'] + [-100] * (m - len(b['labels'])) for b in batch]),
    }

@torch.no_grad()
def generate_sql(model, tok, rows, batch_size=16, max_new_tokens=128):
    prompts = [r['prompt'] for r in rows]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outputs = [None] * len(prompts)
    for s in tqdm(range(0, len(order), batch_size)):
        idx = order[s:s + batch_size]
        batch = tok([prompts[i] for i in idx], return_tensors='pt', padding=True).to('cuda')
        gen = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(gen[:, batch['input_ids'].shape[1]:], skip_special_tokens=True)
        for i, t in zip(idx, texts):
            outputs[i] = clean_generation(t)
    return outputs

In [3]:
 def run_experiment(name, r, alpha, lr, epochs=1):
    set_seed(42)
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='cuda')
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                     target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    model = get_peft_model(model, cfg)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)

    args = TrainingArguments(
        output_dir=f'{PROJECT_DIR}/checkpoints/{name}', num_train_epochs=epochs,
        per_device_train_batch_size=4, gradient_accumulation_steps=4,
        learning_rate=lr, lr_scheduler_type='cosine', warmup_steps=15, fp16=True,
        logging_steps=50, save_strategy='no', report_to='none',
        remove_unused_columns=False, optim='adamw_torch')
    model.config.use_cache = False
    trainer = Trainer(model=model, args=args, train_dataset=train_enc,
                      eval_dataset=val_enc, data_collator=collate)
    trainer.train()
    val_loss = trainer.evaluate()['eval_loss']
    minutes = (time.time() - t0) / 60

    model.save_pretrained(f'{PROJECT_DIR}/adapters/{name}')
    model.config.use_cache = True
    model.eval()
    preds = generate_sql(model, tok, test)
    res = {'name': name, 'r': r, 'alpha': alpha, 'lr': lr, 'epochs': epochs,
           'trainable_params': n_train, 'val_loss': round(val_loss, 4),
           'minutes': round(minutes, 1), **score(preds, test)}
    json.dump(res, open(f'{RESULTS_DIR}/{name}.json', 'w'), indent=2)
    json.dump(preds, open(f'{RESULTS_DIR}/{name}_preds.json', 'w'))
    print(res)
    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    return res

In [4]:
EXPERIMENTS = [
    dict(name='lora_r4',          r=4,  alpha=8,   lr=2e-4),
    dict(name='lora_r64',         r=64, alpha=128, lr=2e-4),
    dict(name='lora_r16_lr5e-5',  r=16, alpha=32,  lr=5e-5),
]
for e in EXPERIMENTS:
    if os.path.exists(f"{RESULTS_DIR}/{e['name']}.json"):
        print('skipping (already done):', e['name'])
        continue
    run_experiment(**e)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step,Training Loss
50,0.166536
100,0.085222
150,0.055993
200,0.051393
250,0.053683
300,0.042648
350,0.040542
400,0.047532
450,0.037415
500,0.033570


Training Loss,Validation Loss,Step
0.033570,0.044188,500


  0%|          | 0/19 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step,Training Loss
50,0.172102
100,0.094261
150,0.060716
200,0.054340
250,0.058900
300,0.047594
350,0.044406
400,0.051947
450,0.041385
500,0.036771


Training Loss,Validation Loss,Step
0.036771,0.051747,500


  0%|          | 0/19 [00:00<?, ?it/s]

{'name': 'lora_r16_lr5e-5', 'r': 16, 'alpha': 32, 'lr': 5e-05, 'epochs': 1, 'trainable_params': 18464768, 'val_loss': 0.0517, 'minutes': 10.4, 'n': 300, 'exact_match': 0.7833333333333333, 'runs_on_sqlite': 0.9766666666666667}


In [5]:
import pandas as pd
rows = []
for f in sorted(glob.glob(f'{RESULTS_DIR}/lora*.json')):
    if f.endswith('_preds.json'):
        continue
    d = json.load(open(f)); d['name'] = os.path.basename(f)[:-5]; rows.append(d)
base = json.load(open(f'{RESULTS_DIR}/baseline.json')); base['name'] = 'baseline (no tuning)'

df = pd.DataFrame([base] + rows)[['name','r','lr','trainable_params','val_loss','exact_match','runs_on_sqlite','minutes']]
df['exact_match'] = (df.exact_match * 100).round(1)
df['runs_on_sqlite'] = (df.runs_on_sqlite * 100).round(1)
df

,name,r,lr,trainable_params,val_loss,exact_match,runs_on_sqlite,minutes
0,baseline (no tuning),NaN,NaN,NaN,NaN,49.7,95.0,NaN
1,lora_r16,16.0,0.00020,NaN,NaN,81.3,98.0,NaN
2,lora_r16_lr5e-5,16.0,0.00005,18464768.0,0.0517,78.3,97.7,10.4
3,lora_r4,4.0,0.00020,4616192.0,0.0442,81.0,97.7,10.5
4,lora_r64,64.0,0.00020,73859072.0,0.0372,82.3,97.3,10.9


In [6]:
!pip uninstall -y torchao
!pip install -q -U transformers accelerate datasets peft bitsandbytes
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, gc, glob
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
PROJECT_DIR = '/content/drive/MyDrive/text2sql-lora'
DATA_DIR = f'{PROJECT_DIR}/data'
RESULTS_DIR = f'{PROJECT_DIR}/results'
sys.path.insert(0, PROJECT_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.7 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import torch
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, set_seed)
from peft import LoraConfig, get_peft_model
from utils import *

MODEL = 'Qwen/Qwen2.5-Coder-7B'
train = load_jsonl(f'{DATA_DIR}/train.jsonl')
val   = load_jsonl(f'{DATA_DIR}/val.jsonl')
test  = load_jsonl(f'{DATA_DIR}/test.jsonl')

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = 'left'

MAX_LEN = 384
N_TRAIN = 4000

def encode(r):
    p = tok(r['prompt'], add_special_tokens=False)['input_ids']
    c = tok(r['completion'], add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
    return {'input_ids': p + c, 'labels': [-100] * len(p) + c}

train_enc = [e for e in map(encode, train[:N_TRAIN]) if len(e['input_ids']) <= MAX_LEN]
val_enc   = [e for e in map(encode, val)             if len(e['input_ids']) <= MAX_LEN]

def collate(batch):
    m = max(len(b['input_ids']) for b in batch)
    pad = tok.pad_token_id
    return {
        'input_ids': torch.tensor([b['input_ids'] + [pad] * (m - len(b['input_ids'])) for b in batch]),
        'attention_mask': torch.tensor([[1] * len(b['input_ids']) + [0] * (m - len(b['input_ids'])) for b in batch]),
        'labels': torch.tensor([b['labels'] + [-100] * (m - len(b['labels'])) for b in batch]),
    }

@torch.no_grad()
def generate_sql(model, tok, rows, batch_size=8, max_new_tokens=128):
    prompts = [r['prompt'] for r in rows]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outputs = [None] * len(prompts)
    for s in tqdm(range(0, len(order), batch_size)):
        idx = order[s:s + batch_size]
        batch = tok([prompts[i] for i in idx], return_tensors='pt', padding=True).to('cuda')
        gen = model.generate(**batch, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(gen[:, batch['input_ids'].shape[1]:], skip_special_tokens=True)
        for i, t in zip(idx, texts):
            outputs[i] = clean_generation(t)
    return outputs

print(len(train_enc), 'train examples')

config.json:   0%|          | 0.00/668 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

4000 train examples


In [8]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
                                             torch_dtype=torch.float16, device_map={'': 0})
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

GPU memory used: 5.6 GB


In [9]:
base_preds = generate_sql(model, tok, test)
base_7b = score(base_preds, test)
print(base_7b)
json.dump({'model': MODEL, '4bit': True, **base_7b}, open(f'{RESULTS_DIR}/baseline_7b.json', 'w'), indent=2)

  0%|          | 0/38 [00:00<?, ?it/s]

{'n': 300, 'exact_match': 0.58, 'runs_on_sqlite': 0.97}


In [10]:
set_seed(42)
model.enable_input_require_grads()
cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                 target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, cfg)
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir=f'{PROJECT_DIR}/checkpoints/qlora_7b', num_train_epochs=1,
    per_device_train_batch_size=2, gradient_accumulation_steps=8, per_device_eval_batch_size=2,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=10, fp16=True,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=10, save_strategy='no', report_to='none',
    remove_unused_columns=False, optim='adamw_torch')

model.config.use_cache = False
t0 = time.time()
trainer = Trainer(model=model, args=args, train_dataset=train_enc,
                  eval_dataset=val_enc, data_collator=collate)
trainer.train()
val_loss = trainer.evaluate()['eval_loss']
minutes = (time.time() - t0) / 60
print('val loss', val_loss, '| minutes', round(minutes, 1))

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


Step,Training Loss
10,0.251414
20,0.109506
30,0.088577
40,0.093184
50,0.053292
60,0.058121
70,0.042575
80,0.053460
90,0.051501
100,0.064460


Training Loss,Validation Loss,Step
0.031536,0.037507,250


val loss 0.03750678151845932 | minutes 38.6


In [11]:
ADAPTER_DIR = f'{PROJECT_DIR}/adapters/qlora_7b'
model.save_pretrained(ADAPTER_DIR)

model.config.use_cache = True
model.eval()
preds = generate_sql(model, tok, test)
tuned = score(preds, test)

res = {'name': 'qlora_7b', 'model': MODEL, 'r': 16, 'alpha': 32, 'lr': 2e-4, 'n_train': len(train_enc),
       'val_loss': round(val_loss, 4), 'minutes': round(minutes, 1), **tuned}
json.dump(res, open(f'{RESULTS_DIR}/qlora_7b.json', 'w'), indent=2)
json.dump(preds, open(f'{RESULTS_DIR}/qlora_7b_preds.json', 'w'))

print(f"{'':20}{'exact match':>13}{'runs on SQLite':>16}")
print(f"{'7B untuned (4-bit)':20}{base_7b['exact_match']:>13.1%}{base_7b['runs_on_sqlite']:>16.1%}")
print(f"{'7B + QLoRA':20}{tuned['exact_match']:>13.1%}{tuned['runs_on_sqlite']:>16.1%}")
print('1.5B + LoRA r=16 for reference: 81.3% / 98.0%')

  0%|          | 0/38 [00:00<?, ?it/s]

                      exact match  runs on SQLite
7B untuned (4-bit)          58.0%           97.0%
7B + QLoRA                  77.3%           97.3%
1.5B + LoRA r=16 for reference: 81.3% / 98.0%
